# BROCHURE GENERATOR

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.
We will be provided a company name and their primary website.

In [1]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_contents, fetch_website_links
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [4]:
links = fetch_website_links("https://www.anthropic.com")
links

['#main',
 '#footer',
 'https://www.anthropic.com/',
 'https://www.anthropic.com/research',
 'https://www.anthropic.com/economic-futures',
 'https://www.anthropic.com/constitution',
 'https://www.anthropic.com/transparency',
 'https://www.anthropic.com/responsible-scaling-policy',
 'http://trust.anthropic.com/',
 'https://www.anthropic.com/learn',
 'https://claude.com/resources/tutorials',
 'https://claude.com/resources/use-cases',
 'https://www.anthropic.com/engineering',
 'https://platform.claude.com/docs',
 'https://www.anthropic.com/company',
 'https://www.anthropic.com/careers',
 'https://www.anthropic.com/events',
 'https://www.anthropic.com/news',
 'https://claude.ai',
 'https://claude.com/product/overview',
 'https://claude.com/product/claude-code',
 'https://claude.com/product/cowork',
 'https://claude.com/product/claude-security',
 'https://claude.com/platform/api',
 'https://claude.com/pricing',
 'https://claude.com/contact-sales',
 'https://www.anthropic.com/claude/opus',
 

In [7]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About Page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [9]:
def get_links_user_prompt(url):
    user_prompt = f"""
    Here is the list of links on the website {url} -
    Please decide which of these are relevant web links for a brochure about the company,
    respond with the full https URL in JSON format.
    Do not include Terms of Service, Privacy, email links.

    Links (some might be relative links)
    """

    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [10]:
print(get_links_user_prompt("https://www.anthropic.com"))


    Here is the list of links on the website https://www.anthropic.com -
    Please decide which of these are relevant web links for a brochure about the company,
    respond with the full https URL in JSON format.
    Do not include Terms of Service, Privacy, email links.

    Links (some might be relative links)
    #main
#footer
https://www.anthropic.com/
https://www.anthropic.com/research
https://www.anthropic.com/economic-futures
https://www.anthropic.com/constitution
https://www.anthropic.com/transparency
https://www.anthropic.com/responsible-scaling-policy
http://trust.anthropic.com/
https://www.anthropic.com/learn
https://claude.com/resources/tutorials
https://claude.com/resources/use-cases
https://www.anthropic.com/engineering
https://platform.claude.com/docs
https://www.anthropic.com/company
https://www.anthropic.com/careers
https://www.anthropic.com/events
https://www.anthropic.com/news
https://claude.ai
https://claude.com/product/overview
https://claude.com/product/claude-

In [11]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            { "role": "system", "content": link_system_prompt },
            { "role": "user", "content": get_links_user_prompt(url) }
        ],
        response_format={ "type": "json_object" }
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links

In [12]:
select_relevant_links("https://www.anthropic.com")

{'links': [{'type': 'about page', 'url': 'https://www.anthropic.com/'},
  {'type': 'company page', 'url': 'https://www.anthropic.com/company'},
  {'type': 'careers page', 'url': 'https://www.anthropic.com/careers'},
  {'type': 'research page', 'url': 'https://www.anthropic.com/research'},
  {'type': 'economic futures page',
   'url': 'https://www.anthropic.com/economic-futures'},
  {'type': 'policy/governance page',
   'url': 'https://www.anthropic.com/constitution'},
  {'type': 'transparency page',
   'url': 'https://www.anthropic.com/transparency'},
  {'type': 'policy page',
   'url': 'https://www.anthropic.com/responsible-scaling-policy'},
  {'type': 'learn/education page', 'url': 'https://www.anthropic.com/learn'},
  {'type': 'engineering page', 'url': 'https://www.anthropic.com/engineering'},
  {'type': 'news page', 'url': 'https://www.anthropic.com/news'},
  {'type': 'events page', 'url': 'https://www.anthropic.com/events'},
  {'type': 'status page', 'url': 'https://status.anthro

In [13]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [14]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 7 relevant links


{'links': [{'type': 'home page', 'url': 'https://huggingface.co'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'GitHub page', 'url': 'https://github.com/huggingface'},
  {'type': 'LinkedIn page',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'}]}

# Step 2: Make the brochure

In [15]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [16]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 20 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
bytedance-research/Lance
Updated
2 days ago
•
1k
•
615
Supertone/supertonic-3
Updated
4 days ago
•
37.5k
•
555
openbmb/MiniCPM-V-4.6
Updated
3 days ago
•
222k
•
897
SulphurAI/Sulphur-2-base
Updated
about 12 hours ago
•
1.25M
•
1.25k
tencent/Hy-MT2-1.8B
Updated
about 8 hours ago
•
564
•
261
Bro

In [17]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [18]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
    You are looking at a company called: {company_name}
    Here are the contents of it's landing page and other relevant pages;
    use this information to build a short brochure of the company in markdown without code blocks.\n\n
    """
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [19]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 13 relevant links


"\n    You are looking at a company called: HuggingFace\n    Here are the contents of it's landing page and other relevant pages;\n    use this information to build a short brochure of the company in markdown without code blocks.\n\n\n    ## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nbytedance-research/Lance\nUpdated\n2 days ago\n•\n1k\n•\n615\nSupertone/supertonic-3\nUpdated\n4 days ago\n•\

In [20]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            { "role": "system", "content": brochure_system_prompt },
            { "role": "user", "content": get_brochure_user_prompt(company_name, url) }
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [21]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 3 relevant links


# Hugging Face - Revolutionizing AI Together

## Who We Are
Hugging Face is a privately held company founded in 2016 and is dedicated to building the future of artificial intelligence through community collaboration. We specialize in machine learning, natural language processing (NLP), and deep learning. Our platform offers the tools needed for the machine learning community to collaborate on models, datasets, and applications, facilitating a rich ecosystem of AI innovation.

## What We Offer
- **Models**: Access to over 2 million models spanning various domains.
- **Datasets**: Over 500,000 datasets for machine learning projects.
- **Spaces**: A platform to explore and host machine learning applications.
- **Enterprise Solutions**: Tailored enterprise support, storage solutions, and inference endpoints.

## Our Customers
Hugging Face serves a diverse range of customers, from individual developers to large enterprises. We are committed to democratizing AI, making our technology accessible to everyone, so you can create, discover, and collaborate on machine learning with ease.

## Careers at Hugging Face
Join our fast-growing team of innovators! Our culture is centered around collaboration, open communication, and continuous learning. We are looking for talented individuals who share our passion for AI and who thrive in a dynamic and inclusive environment. Explore current openings and find your place in our mission to empower developers and organizations alike.

## Company Culture
At Hugging Face, we foster a vibrant, inclusive, and creative workplace. Our team is driven by a shared vision of building innovative AI solutions and supporting one another through challenges. We believe in the power of community and encourage our members to share knowledge and insights with each other.

## Join Us in Shaping the Future of AI
Whether you are a potential customer looking for AI solutions, an investor wanting to be a part of an exciting venture, or a talent eager to contribute to groundbreaking technology, Hugging Face invites you to engage with us on this incredible journey. 

**Explore more at [Hugging Face](https://huggingface.co)**

In [22]:
# Now, It's a small improvement (with a small adjustment, we can change this so that the results stream back from OpenAI, with the familiar typewriter animation)

def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            { "role": "system", "content": brochure_system_prompt },
            { "role": "user", "content": get_brochure_user_prompt(company_name, url) }
        ],
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [23]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 8 relevant links


# Hugging Face Brochure

## About Us
Hugging Face is more than just a company; it's a thriving AI community dedicated to constructing the future of machine learning. Our platform enables collaboration on models, datasets, and applications, empowering everyone from researchers to companies to push the boundaries of what's possible in AI.

## Our Offerings
- **Models**: Explore over 2 million state-of-the-art models available for all kinds of AI tasks.
- **Datasets**: Gain access to a rich repository of more than 500,000 datasets, facilitating robust training and evaluation of machine learning models.
- **Spaces**: Utilizing our innovative workspaces, users can run and share applications seamlessly.
- **Enterprise Solutions**: We provide tailored support for larger organizations, including the Hugging Face PRO and enterprise support systems.

## Community Spirit
At Hugging Face, we foster a vibrant community through various avenues:
- **Forums and Discord**: Engage with fellow enthusiasts and experts.
- **Blog and Daily Papers**: Stay updated with the latest in research and community activities.
- **Collaboration**: We believe in open-source collaboration and provide tools for hosting public models and applications.

## Company Culture
Our culture is rooted in innovation, inclusivity, and collaboration. We value diverse perspectives and encourage creativity, making it a stimulating environment for those passionate about AI and machine learning. Our team operates with a sense of openness and shared vision, where each member's contribution is respected and celebrated.

## Career Opportunities
Join our mission to revolutionize the AI landscape! At Hugging Face, we welcome passionate individuals from various backgrounds:
- Work across different roles in a dynamic environment focused on growth and learning.
- Contribute directly to groundbreaking projects in AI and machine learning.
- Enjoy a flexible work culture that prioritizes your well-being and professional development.

## Who We Serve
Our diverse clientele ranges from individual developers to large enterprises looking for advanced AI solutions. Organizations across sectors utilize our models and datasets to enhance their capabilities, demonstrating the versatility and impact of our offerings.

## Connect with Us
Be a part of the Hugging Face revolution! Whether you’re a prospective customer, investor, or recruit, we invite you to join our community and embark on this exciting journey into the future of AI.

Learn more at [huggingface.co](https://huggingface.co).